# DreamCoder Train/Test Pipeline – Schritt für Schritt

Dieses Notebook führt die Pipeline **nicht als ein großes `run_all.sh`** aus, sondern jeden Schritt einzeln. Nach jedem Schritt kannst du die erzeugten Dateien prüfen, bevor du weitergehst.

Wichtig: Dieses Notebook muss im Ordner `dreamcoder_train_test_pipeline_step_by_step` oder `dreamcoder_train_test_pipeline_robust` liegen und mit dem Kernel `Python (dreamcoder)` ausgeführt werden.


In [ ]:
from pathlib import Path
import os, subprocess, json, pandas as pd, textwrap

ROOT = Path.cwd()
print("Current folder:", ROOT)
print("Files:", sorted([p.name for p in ROOT.iterdir()]))

assert (ROOT / "scripts").exists(), "Der Ordner scripts fehlt. Öffne das Notebook im Pipeline-Hauptordner."
assert (ROOT / "dataset").exists(), "Der Ordner dataset fehlt."


## 0. Konfiguration

Hier stellst du ein, welcher Train-/Test-Datensatz benutzt wird und wie stark DreamCoder suchen soll.


In [ ]:
TRAIN_DATASET_FILE = "dataset/T=2_train.json"
TEST_DATASET_FILE = "dataset/T=2_test.json"
DREAMCODER_REPO_ROOT = "/mnt/c/BA/ec"
PYTHON = "/root/miniconda3/envs/dreamcoder/bin/python"

# Voller Datensatz + Consolidation an — entspricht dem Lauf mit 26 % Accuracy
MAX_TRAIN_TASKS = 0          # 0 = alle 2339 Trainings-Tasks (statt nur 150)
DREAMCODER_TIMEOUT = 15
DREAMCODER_TESTING_TIMEOUT = 15
DREAMCODER_ITERATIONS = 4
DREAMCODER_FRONTIER_SIZE = 10
DREAMCODER_USE_RECOGNITION = "false"
DREAMCODER_NO_CONSOLIDATION = "false"   # false = Consolidation AN
DREAMCODER_CPUS = 4

train_path = ROOT / TRAIN_DATASET_FILE
test_path = ROOT / TEST_DATASET_FILE
print("Train:", train_path, train_path.exists())
print("Test :", test_path, test_path.exists())

if not train_path.exists():
    print("FEHLT:", train_path)
    print("Kopiere z. B. im Terminal:")
    print(f"cp /mnt/c/BA/deepcoder/dataset/T=2_train.json {train_path}")

assert test_path.exists(), "Testdatensatz fehlt."
assert train_path.exists(), "Trainingsdatensatz fehlt."


## 0.1 Output-Ordner erzeugen

Alle Ergebnisse dieses Laufs werden in `outputs/<RUN_KEY>/` gespeichert.


In [ ]:
def sanitize(name: str) -> str:
    return Path(name).stem.replace("=", "_").replace(".", "_").replace("-", "_")

TRAIN_KEY = sanitize(TRAIN_DATASET_FILE)
TEST_KEY = sanitize(TEST_DATASET_FILE)
RUN_KEY = (
    f"train_{TRAIN_KEY}__test_{TEST_KEY}__"
    f"ET_{DREAMCODER_TIMEOUT}_TT_{DREAMCODER_TESTING_TIMEOUT}_"
    f"it_{DREAMCODER_ITERATIONS}_MF_{DREAMCODER_FRONTIER_SIZE}_"
    f"rec_{DREAMCODER_USE_RECOGNITION}_nocons_{DREAMCODER_NO_CONSOLIDATION}"
)
OUTPUT_ROOT = ROOT / "outputs" / RUN_KEY
LOG_DIR = OUTPUT_ROOT / "logs"

step_dirs = [
    "step01_validate_datasets",
    "step02_convert_train_test",
    "step03_create_task_pickles",
    "step04_run_dreamcoder",
    "step05_detect_operations",
    "step06_normalize_programs",
    "step07_calculate_metrics",
    "step08_summarize_results",
]
for d in step_dirs:
    (OUTPUT_ROOT / d).mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print("RUN_KEY:", RUN_KEY)
print("OUTPUT_ROOT:", OUTPUT_ROOT)


## Hilfsfunktion zum Ausführen eines Schritts


In [ ]:
def run_step(name, cmd, log_name=None):
    log_name = log_name or f"{name}.log"
    log_path = LOG_DIR / log_name
    print("\n" + "="*80)
    print(name)
    print("Command:", " ".join(map(str, cmd)))
    print("Log:", log_path)
    print("="*80 + "\n")
    with open(log_path, "w", encoding="utf-8") as f:
        process = subprocess.Popen(
            list(map(str, cmd)),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in process.stdout:
            print(line, end="")
            f.write(line)
        rc = process.wait()
    print("\nReturn code:", rc)
    if rc != 0:
        raise RuntimeError(f"{name} failed with return code {rc}")
    return log_path


## Schritt 1 – Datensätze validieren

Prüft Anzahl Aufgaben, Beispiele, Output-Typen, Null-Outputs usw.

Ergebnisse: `step01_validate_datasets/`


In [ ]:
run_step("step01_validate_datasets", [
    PYTHON,
    ROOT / "scripts" / "step01_validate_datasets.py",
    train_path,
    test_path,
    OUTPUT_ROOT / "step01_validate_datasets",
])

print("\nErzeugte Dateien:")
for p in sorted((OUTPUT_ROOT / "step01_validate_datasets").iterdir()): print(p)


## Schritt 2 – Train/Test-Aufgaben konvertieren

Konvertiert JSON-Aufgaben in ein Zwischenformat mit Task-Namen, Request-Typen, Beispielen und Referenzprogrammen.

Ergebnisse: `step02_convert_train_test/`


In [ ]:
run_step("step02_convert_train_test_tasks", [
    PYTHON,
    ROOT / "scripts" / "step02_convert_train_test_tasks.py",
    train_path,
    test_path,
    OUTPUT_ROOT / "step02_convert_train_test",
    MAX_TRAIN_TASKS,
])

print("\nErzeugte Dateien:")
for p in sorted((OUTPUT_ROOT / "step02_convert_train_test").iterdir()): print(p)


## Schritt 3 – DreamCoder Task-Pickles erzeugen

Erzeugt echte DreamCoder-`Task`-Objekte für Training und Test.

Ergebnisse: `step03_create_task_pickles/`


In [ ]:
run_step("step03_create_dreamcoder_task_pickles", [
    PYTHON,
    ROOT / "scripts" / "step03_create_dreamcoder_task_pickles.py",
    OUTPUT_ROOT / "step02_convert_train_test" / "step02_train_tasks.json",
    OUTPUT_ROOT / "step02_convert_train_test" / "step02_test_tasks.json",
    OUTPUT_ROOT / "step03_create_task_pickles",
    DREAMCODER_REPO_ROOT,
])

print("\nErzeugte Dateien:")
for p in sorted((OUTPUT_ROOT / "step03_create_task_pickles").iterdir()): print(p)


## Schritt 4 – DreamCoder Training/Test ausführen

Das ist der rechenintensive Schritt. DreamCoder sucht Programme für Trainings- und Testaufgaben. Die robuste Version speichert auch `step04_stdout.log` und extrahiert Test-HITs aus dem stdout, falls die Test-Frontiers nicht im Ergebnisobjekt stehen.

Ergebnisse: `step04_run_dreamcoder/`


In [ ]:
run_step("step04_run_dreamcoder_train_test", [
    PYTHON,
    ROOT / "scripts" / "step04_run_dreamcoder_train_test.py",
    OUTPUT_ROOT / "step03_create_task_pickles" / "step03_train_tasks.pkl",
    OUTPUT_ROOT / "step03_create_task_pickles" / "step03_test_tasks.pkl",
    OUTPUT_ROOT / "step02_convert_train_test" / "step02_train_tasks.json",
    OUTPUT_ROOT / "step02_convert_train_test" / "step02_test_tasks.json",
    OUTPUT_ROOT / "step04_run_dreamcoder",
    DREAMCODER_REPO_ROOT,
    DREAMCODER_TIMEOUT,
    DREAMCODER_TESTING_TIMEOUT,
    DREAMCODER_ITERATIONS,
    DREAMCODER_FRONTIER_SIZE,
    DREAMCODER_USE_RECOGNITION,
    DREAMCODER_NO_CONSOLIDATION,
    DREAMCODER_CPUS,
])

print("\nErzeugte Dateien:")
for p in sorted((OUTPUT_ROOT / "step04_run_dreamcoder").iterdir()): print(p)


In [ ]:
from pathlib import Path

log_path = LOG_DIR / "step04_run_dreamcoder_train_test.log"

print("Log-Datei:", log_path)
print("=" * 80)

text = log_path.read_text(encoding="utf-8", errors="replace")
print("\n".join(text.splitlines()[-120:]))

## Schritt 4.1 – Test-HITs prüfen

Hier prüfst du direkt, ob `step04_test_results.csv` gelöste Testaufgaben enthält.


In [ ]:
test_csv = OUTPUT_ROOT / "step04_run_dreamcoder" / "step04_test_results.csv"
summary_json = OUTPUT_ROOT / "step04_run_dreamcoder" / "step04_train_test_summary.json"
stdout_hits = OUTPUT_ROOT / "step04_run_dreamcoder" / "step04_stdout_test_hits.csv"

print("test_csv:", test_csv, test_csv.exists())
print("stdout_hits:", stdout_hits, stdout_hits.exists())

if test_csv.exists():
    df = pd.read_csv(test_csv)
    print("Rows:", len(df))
    print("Solved:", int(df.get("solved", pd.Series(dtype=int)).fillna(False).astype(bool).sum()))
    display(df.head(20))

if stdout_hits.exists():
    hits_df = pd.read_csv(stdout_hits)
    print("stdout HIT rows:", len(hits_df))
    display(hits_df.head(20))

if summary_json.exists():
    print(json.dumps(json.loads(summary_json.read_text()), indent=2))


## Schritt 5 – Operationen erkennen

Prüft, welche Referenzoperationen und Lösungstokens vorkommen und ob unbekannte Tokens existieren.

Ergebnisse: `step05_detect_operations/`


In [ ]:
run_step("step05_detect_operations", [
    PYTHON,
    ROOT / "scripts" / "step05_detect_operations.py",
    OUTPUT_ROOT / "step04_run_dreamcoder" / "step04_test_results.csv",
    OUTPUT_ROOT / "step05_detect_operations",
])

print("\nErzeugte Dateien:")
for p in sorted((OUTPUT_ROOT / "step05_detect_operations").iterdir()): print(p)


## Schritt 6 – Programme normalisieren

Normalisiert Referenzprogramme und DreamCoder-Lösungen auf gemeinsame Tokenebenen. Ungelöste Aufgaben werden nicht als triviale Lösungen gezählt.

Ergebnisse: `step06_normalize_programs/`


In [ ]:
run_step("step06_normalize_programs", [
    PYTHON,
    ROOT / "scripts" / "step06_normalize_programs.py",
    OUTPUT_ROOT / "step04_run_dreamcoder" / "step04_test_results.csv",
    OUTPUT_ROOT / "step06_normalize_programs",
])

print("\nErzeugte Dateien:")
for p in sorted((OUTPUT_ROOT / "step06_normalize_programs").iterdir()): print(p)


## Schritt 7 – Metriken berechnen

Berechnet Accuracy und Programmmetriken.

Ergebnisse: `step07_calculate_metrics/`


In [ ]:
run_step("step07_calculate_metrics", [
    PYTHON,
    ROOT / "scripts" / "step07_calculate_metrics.py",
    OUTPUT_ROOT / "step06_normalize_programs" / "step06_normalized_test_programs.csv",
    OUTPUT_ROOT / "step07_calculate_metrics",
])

print("\nErzeugte Dateien:")
for p in sorted((OUTPUT_ROOT / "step07_calculate_metrics").iterdir()): print(p)
print("\nMetrics summary:")
print((OUTPUT_ROOT / "step07_calculate_metrics" / "step07_metrics_summary.json").read_text())


## Schritt 8 – Zusammenfassung erzeugen

Schreibt die finale Zusammenfassung und trennt gelöste/ungelöste Testaufgaben.

Ergebnisse: `step08_summarize_results/`


In [ ]:
run_step("step08_summarize_results", [
    PYTHON,
    ROOT / "scripts" / "step08_summarize_results.py",
    OUTPUT_ROOT / "step07_calculate_metrics" / "step07_test_results_with_metrics.csv",
    OUTPUT_ROOT / "step07_calculate_metrics" / "step07_metrics_summary.json",
    OUTPUT_ROOT / "step04_run_dreamcoder" / "step04_train_test_summary.json",
    OUTPUT_ROOT / "step08_summarize_results",
])

summary = OUTPUT_ROOT / "step08_summarize_results" / "step08_summary.txt"
print("\nFinal Summary:")
print(summary)
print(summary.read_text())


## Ergebnisdateien

Die wichtigsten Dateien dieses Laufs:


In [ ]:
important = [
    OUTPUT_ROOT / "logs",
    OUTPUT_ROOT / "step04_run_dreamcoder" / "step04_stdout.log",
    OUTPUT_ROOT / "step04_run_dreamcoder" / "step04_stdout_test_hits.csv",
    OUTPUT_ROOT / "step04_run_dreamcoder" / "step04_test_results.csv",
    OUTPUT_ROOT / "step06_normalize_programs" / "step06_normalized_test_programs.csv",
    OUTPUT_ROOT / "step07_calculate_metrics" / "step07_metrics_summary.json",
    OUTPUT_ROOT / "step08_summarize_results" / "step08_summary.txt",
    OUTPUT_ROOT / "step08_summarize_results" / "dreamcoder_test_results_complete.csv",
    OUTPUT_ROOT / "step08_summarize_results" / "dreamcoder_test_solved_tasks.csv",
    OUTPUT_ROOT / "step08_summarize_results" / "dreamcoder_test_unsolved_tasks.csv",
]
for p in important:
    print(p, "EXISTS" if p.exists() else "MISSING")


In [ ]:
# ============================================================
# Letzte Zelle: Diagramm der finalen Metriken erzeugen
# Robuste Version: ignoriert abgebrochene Läufe
# ============================================================

from pathlib import Path
import json
import matplotlib.pyplot as plt

try:
    ROOT
except NameError:
    ROOT = Path.cwd()

outputs_dir = ROOT / "outputs"

if not outputs_dir.exists():
    raise FileNotFoundError(f"Output-Ordner nicht gefunden: {outputs_dir}")

# Nur Läufe verwenden, bei denen step07_metrics_summary.json existiert
valid_runs = []

for run_dir in outputs_dir.iterdir():
    if not run_dir.is_dir():
        continue

    metrics_file = run_dir / "step07_calculate_metrics" / "step07_metrics_summary.json"

    if metrics_file.exists():
        valid_runs.append(run_dir)

if not valid_runs:
    raise FileNotFoundError(
        "Kein vollständiger Lauf mit step07_metrics_summary.json gefunden."
    )

# Neuesten vollständigen Lauf nehmen
latest_run = sorted(
    valid_runs,
    key=lambda p: p.stat().st_mtime,
    reverse=True
)[0]

metrics_path = latest_run / "step07_calculate_metrics" / "step07_metrics_summary.json"
summary_path = latest_run / "step08_summarize_results" / "step08_summary.txt"

print("Verwendeter vollständiger Ergebnisordner:")
print(latest_run)
print()

with open(metrics_path, "r", encoding="utf-8") as f:
    metrics = json.load(f)

# Werte robust lesen
accuracy = (
    metrics.get("test_accuracy")
    or metrics.get("accuracy")
    or metrics.get("Accuracy")
    or 0.0
)

operation_score = (
    metrics.get("normalized_operation_score_mean")
    or metrics.get("operation_score_mean")
    or 0.0
)

position_score = (
    metrics.get("normalized_position_score_mean")
    or metrics.get("position_score_mean")
    or 0.0
)

order_score = (
    metrics.get("normalized_order_score_mean")
    or metrics.get("order_score_mean")
    or 0.0
)

edit_score = (
    metrics.get("normalized_edit_score_mean")
    or metrics.get("edit_score_mean")
    or 0.0
)

labels = [
    "Accuracy",
    "Operation",
    "Position",
    "Reihenfolge",
    "Edit"
]

values = [
    accuracy,
    operation_score,
    position_score,
    order_score,
    edit_score
]

plt.figure(figsize=(9, 5))
bars = plt.bar(labels, values)

plt.ylim(0, 1)
plt.ylabel("Durchschnittlicher Score")
plt.xlabel("Metrik")
plt.title("Durchschnittliche Metrikwerte – alle Testaufgaben")

for bar, value in zip(bars, values):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.02,
        f"{value:.3f}",
        ha="center",
        va="bottom"
    )

plt.xticks(rotation=30, ha="right")
plt.tight_layout()

plot_path = latest_run / "step08_summarize_results" / "metric_bar_chart_all_tasks.png"
plt.savefig(plot_path, dpi=200, bbox_inches="tight")

plt.show()

print()
print("Diagramm gespeichert unter:")
print(plot_path)

if summary_path.exists():
    print()
    print("=" * 80)
    print("Finale Summary")
    print("=" * 80)
    print(summary_path.read_text(encoding="utf-8"))

## Optional: mittlerer Lauf

Erst ausführen, wenn der kleine Lauf funktioniert. Ändere oben die Konfiguration auf z. B.:

```python
MAX_TRAIN_TASKS = 300
DREAMCODER_TIMEOUT = 15
DREAMCODER_TESTING_TIMEOUT = 15
DREAMCODER_ITERATIONS = 2
DREAMCODER_FRONTIER_SIZE = 30
DREAMCODER_USE_RECOGNITION = "true"
DREAMCODER_NO_CONSOLIDATION = "true"
```

Dann Notebook-Kernel neu starten und wieder ab Schritt 0 ausführen.
